# 03 — Active-View Grasping Pipeline (UGAP vs. single-shot baseline)

The main experiment: for each test object (opaque, transparent,
reflective, thin), compare

- **Baseline:** reconstruct once from a fixed set of views, detect grasps,
  execute the top-scored grasp regardless of uncertainty.
- **UGAP:** reconstruct, compute evidential uncertainty, if any
  candidate-grasp region is above the uncertainty threshold take an
  extra next-best-view and re-reconstruct (up to `max_views` times), only
  then detect and execute a grasp -- and only accept grasps whose region
  uncertainty clears the threshold.

Metric: simulated grasp success rate (via PyBullet contact/lift check) and
number of views used, broken down by object class. The gap should be
largest on transparent/reflective/thin objects -- that's the headline
result.

Run notebooks 00-02 first (needs the trained uncertainty head checkpoint).


In [ ]:
import sys
sys.path.append('.')
import numpy as np
import torch
import glob, os

from ugap.reconstruction import DUSt3RWrapper
from ugap.uncertainty import EvidentialUncertaintyHead
from ugap.nbv import NextBestViewPlanner
from ugap.grasping import GRConvNetWrapper
from ugap.utils.pointcloud_utils import build_evidential_features
from ugap.utils.visualization import plot_view_coverage
from ugap.data.sim_generator import generate_dataset


## Step 1: test set

Generate a held-out set of objects (different seed from training in
notebook 02) across all four classes.


In [ ]:
generate_dataset(
    out_root='data/sim_sequences_test',
    n_objects_per_class=5,
    n_views=3,   # deliberately FEW initial views -- forces the NBV loop to matter
    image_size=256,
    seed=123,
    overwrite=True,
)
test_object_dirs = sorted(glob.glob('data/sim_sequences_test/obj_*'))
print(f"{len(test_object_dirs)} test objects")


**Note on grasp detector:** this uses GR-ConvNet (trained on Cornell, optionally Jacquard) rather than GraspNet-baseline -- see README for why. It predicts a 2D grasp rectangle on one RGB-D view, which we lift into 3D using that view's depth + intrinsics before applying the same uncertainty gate.


## Step 2: load trained components

In [ ]:
grasp_wrapper = GRConvNetWrapper(
    checkpoint='checkpoints/grconvnet/cornell_rgbd_iou_0.96',
    input_size=224,
    device='cuda',
)


## Step 3: helper functions wiring everything together

In [ ]:
def reconstruct_fn(image_paths):
    return recon_wrapper.reconstruct(image_paths)

def uncertainty_fn(recon):
    n_views_observed = np.full(len(recon.points), len(recon.image_paths))
    features = build_evidential_features(
        recon.points, recon.colors, recon.confidence,
        recon.per_image_poses, n_views_observed,
    )
    with torch.no_grad():
        feats_t = torch.from_numpy(features).float().to('cuda')
        gamma, nu, alpha, beta = uncertainty_model(feats_t)
        _, epistemic = uncertainty_model.uncertainty_from_params(nu, alpha, beta)
    return recon.points, epistemic.cpu().numpy()

def capture_fn_factory(obj_dir, available_views):
    """
    Simulates 'capturing a new view': since sim_generator already rendered
    a fixed ring of views for this object, we approximate NBV by picking
    the closest not-yet-used pre-rendered view to the planner's proposed
    direction. Swap this for a live PyBullet camera render (using
    candidate.position/look_at directly) once you want continuous NBV
    rather than a fixed ring -- straightforward since sim_generator.py's
    _render_views already shows the PyBullet camera-render call.
    """
    used = set(available_views)
    all_ring_views = sorted(glob.glob(os.path.join(obj_dir, 'views', 'view_*.png')))

    def capture_fn(candidate):
        remaining = [v for v in all_ring_views if v not in used]
        if not remaining:
            return all_ring_views[-1]  # nothing left, reuse last
        choice = remaining[0]  # TODO: pick by angular closeness to candidate.position
        used.add(choice)
        return choice

    return capture_fn


## Step 4: run baseline vs. UGAP on every test object

In [ ]:
def run_ugap(obj_dir, n_initial_views=3):
    image_paths = sorted(glob.glob(os.path.join(obj_dir, 'views', 'view_*.png')))[:n_initial_views]
    capture_fn = capture_fn_factory(obj_dir, image_paths)

    recon, epistemic, extra_views = nbv_planner.run_active_loop(
        reconstruct_fn, uncertainty_fn, capture_fn, image_paths,
    )

    ref = _pick_reference_view(recon)
    rgb = (recon.rgb_images[ref] * 255).astype('uint8')
    depth = recon.depth_maps[ref]

    candidates = grasp_wrapper.detect_grasps(rgb, depth, n_grasps=5)
    for c in candidates:
        grasp_wrapper.lift_to_3d(c, depth, recon.intrinsics[ref], recon.per_image_poses[ref])

    candidates = grasp_wrapper.gate_by_uncertainty(candidates, recon.points, epistemic)
    best = grasp_wrapper.best_accepted_grasp(candidates)  # gated: only uncertainty-cleared grasps
    return {"n_views_used": n_initial_views + extra_views, "grasp": best}


In [ ]:
# NOTE: grasp *success* here means "the pipeline returned a confident,
# accepted grasp" as a proxy metric. For a real success/failure signal,
# execute `best.translation`/`best.rotation` in PyBullet (load the object
# with its known mass/friction, command the gripper, then check contact +
# lift height > threshold) -- straightforward to add once you're ready to
# move past the plumbing/pilot stage.

results = []
for obj_dir in test_object_dirs:
    object_class = open(os.path.join(obj_dir, 'object_class.txt')).read().strip()

    baseline = run_baseline(obj_dir)
    ugap = run_ugap(obj_dir)

    results.append({
        "object": os.path.basename(obj_dir),
        "class": object_class,
        "baseline_grasp_found": baseline["grasp"] is not None,
        "baseline_views_used": baseline["n_views_used"],
        "ugap_grasp_found": ugap["grasp"] is not None,
        "ugap_views_used": ugap["n_views_used"],
    })

import pandas as pd
df = pd.DataFrame(results)
df


## Step 5: summarize by object class

This is the table/figure you'd put in the paper: does UGAP's
uncertainty-gated grasp acceptance rate beat the baseline specifically on
transparent/reflective/thin objects, at the cost of a few extra views?


In [ ]:
summary = df.groupby('class').agg(
    baseline_success_rate=('baseline_grasp_found', 'mean'),
    ugap_success_rate=('ugap_grasp_found', 'mean'),
    avg_baseline_views=('baseline_views_used', 'mean'),
    avg_ugap_views=('ugap_views_used', 'mean'),
)
summary


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
summary[['baseline_success_rate', 'ugap_success_rate']].plot(kind='bar', ax=ax)
ax.set_ylabel('Confident-grasp rate')
ax.set_title('Baseline vs. UGAP, by object class')
plt.tight_layout()
plt.show()


## Next steps once this plumbing runs end-to-end

1. Replace the analytic placeholder `target_error` in notebook 02 with
   real ground-truth-driven error (closed-form primitive distance, or
   ClearGrasp depth maps) before trusting any calibration numbers.
2. Replace the fixed-ring `capture_fn` here with a live PyBullet camera
   render at the planner's actual proposed pose for true continuous NBV.
3. Add the PyBullet contact/lift check for a real grasp-success signal
   instead of the "confident grasp found" proxy.
4. Once the pipeline is validated on sim data, run the same script against
   a small ClearGrasp/TOD subset for real-image numbers.
